# What You Give Up: Interpretability

**Computational Text Analysis Pack · Module 28 — The Neural Critique**

*Technique:* Coefficients as evidence vs opaque models; what a footnote can cite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/15-the-neural-critique/28a_interpretability_lost.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Module 7c read a linear classifier's coefficients as evidence — which words
pushed a decision, and how hard. That is a humanities-usable output: you can
quote it, argue with it, and put it in a footnote.

This notebook asks what the equivalent is for a neural model, and finds that
there isn't one.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sub = df[(df["language"] == "English") & (df["n_words"] >= 30)].reset_index(drop=True)
tr, te = train_test_split(np.arange(len(sub)), test_size=0.3, random_state=0,
                          stratify=sub["category"])
vec = TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=8000)
A = vec.fit_transform(sub["text"].iloc[tr]); B = vec.transform(sub["text"].iloc[te])
clf = LogisticRegression(max_iter=1000).fit(A, sub["category"].iloc[tr])
print(f"linear model accuracy: {accuracy_score(sub['category'].iloc[te], clf.predict(B)):.3f}")

vocab = np.array(vec.get_feature_names_out())
for i, cat in enumerate(clf.classes_[:3]):
    top = np.argsort(clf.coef_[i])[::-1][:10]
    print(f"\n{cat}: " + ", ".join(f"{vocab[j]} ({clf.coef_[i][j]:+.2f})" for j in top[:7]))

## That table is a citable object

Each number is a weight in a model you can write down in closed form. If a
reviewer asks *why did it classify this document as health*, you can answer
exactly, and the answer does not change between runs.

Now consider what the neural equivalent would be. The LaBSE classifier in module
23d has 768 input dimensions, and **none of them is a word**. There is no row to
quote.

In [ ]:
bank = np.load("labse_headlines.npy")
index = pd.read_csv("labse_headlines_index.csv")
ok = index["category"].notna().values
Xe, ye = bank[ok], index.loc[ok, "category"].values
etr, ete = train_test_split(np.arange(len(Xe)), test_size=0.3, random_state=0, stratify=ye)
eclf = LogisticRegression(max_iter=2000).fit(Xe[etr], ye[etr])
print(f"embedding model accuracy: {accuracy_score(ye[ete], eclf.predict(Xe[ete])):.3f}")
print(f"\nits 'most important features' for {eclf.classes_[0]}:")
top = np.argsort(eclf.coef_[0])[::-1][:8]
print("   " + ", ".join(f"dim_{j} ({eclf.coef_[0][j]:+.2f})" for j in top))
print()
print("dim_412 is not a word. It is a coordinate in a space produced by a model")
print("trained on text you have never seen. There is nothing here to cite.")

## The usual answers, and why they are partial

**Attention weights.** Module 25a already flagged this: Jain and Wallace (2019)
showed they can be altered without changing predictions. Describe them as where
information flowed, not why a decision was made.

**LIME and SHAP.** These fit a *simple, interpretable model locally* around one
prediction. Useful — but you are then citing the explanation model, not the
model that made the decision, and the two can disagree.

**Probing.** Train a classifier to detect whether some property is encoded in
the representation. Tells you information is present; not that the model used it.

None recovers what the coefficient table gave for free.

In [ ]:
# Occlusion: the honest, model-agnostic version. Remove a word, see what moves.
doc = sub["text"].iloc[te[0]]
toks = tokenise(doc)[:40]
base = clf.predict_proba(vec.transform([" ".join(toks)]))[0].max()
drops = []
for i in range(len(toks)):
    without = toks[:i] + toks[i+1:]
    p = clf.predict_proba(vec.transform([" ".join(without)]))[0].max()
    drops.append((toks[i], base - p))
drops.sort(key=lambda t: -abs(t[1]))
print(f"baseline confidence {base:.3f}; removing each word changes it by:\n")
for w, d in drops[:10]:
    print(f"   {w:<18}{d:+.4f}")
print()
print("This works on ANY model, including a transformer — but it costs one")
print("forward pass per word, and it explains one document at a time. It does")
print("not give you the global statement a coefficient table does.")

### 🔧 Your turn 1

Run the occlusion loop against a document your linear model gets wrong. Does it show you why?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Usually yes, and usefully — a misclassification often traces to one or two
words doing too much work, which is a diagnosis you can act on. That is the
argument for keeping an interpretable model in the pipeline even when a neural
one scores higher: you use the linear model to understand the corpus and the
neural model to process it, and you report both. A study whose only artefact is
an accuracy number has produced a system, not a reading.

</details>